# 📺 The Simpsons — OLS y Corrección de Heterocedasticidad

**Flujo:**
1. OLS estándar
2. Test de Breusch-Pagan → detectar heterocedasticidad
3. Si hay heterocedasticidad → OLS con errores robustos HC3 (White)
4. WLS (Mínimos Cuadrados Ponderados) como alternativa
5. Comparativa de los 3 modelos con 3 splits (80/20, 70/30, 60/40)
6. Summary final

**Variable objetivo:** `imdb_rating`  
**Predictores:** `viewers_millions`, `season`, `duration_min`, `number_in_season`

---

## 0. Imports y configuración

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
from statsmodels.stats.stattools import durbin_watson
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from config.database import engine

os.makedirs("../data/graficas", exist_ok=True)
GRAFICAS = "../data/graficas"

plt.rcParams.update({"figure.figsize": (12, 5), "axes.grid": True, "grid.alpha": 0.3})
sns.set_theme(style="whitegrid", palette="muted")
print("✅ Imports OK")

## 1. Carga de datos desde PostgreSQL

In [ ]:
df = pd.read_sql("""
    SELECT episode_id, viewers_millions, imdb_rating,
           duration_min, season, number_in_season
    FROM fact_ratings
    ORDER BY episode_id
""", engine)

FEATURES = ["viewers_millions", "season", "duration_min", "number_in_season"]
TARGET   = "imdb_rating"

df_model = df[FEATURES + [TARGET]].dropna()
X_raw = df_model[FEATURES]
y     = df_model[TARGET]

print(f"Total registros: {len(df_model)}")
print(f"\nEstadísticas descriptivas:")
df_model.describe().round(4)

## 2. OLS estándar — modelo base (datos completos)

In [ ]:
X_const = sm.add_constant(X_raw)
modelo_ols = sm.OLS(y, X_const).fit()
print(modelo_ols.summary())

## 3. Diagnóstico de heterocedasticidad

In [ ]:
residuos = modelo_ols.resid
y_pred   = modelo_ols.fittedvalues

# ── Test de Breusch-Pagan ─────────────────────────────────────────────────────
bp_lm, bp_p, bp_f, bp_fp = het_breuschpagan(residuos, X_const)

# ── Test de White ─────────────────────────────────────────────────────────────
w_lm, w_p, w_f, w_fp = het_white(residuos, X_const)

# ── Durbin-Watson ─────────────────────────────────────────────────────────────
dw = durbin_watson(residuos)

# ── Shapiro-Wilk ─────────────────────────────────────────────────────────────
muestra = residuos.sample(min(500, len(residuos)), random_state=42)
sw_stat, sw_p = stats.shapiro(muestra)

print("=" * 60)
print("  DIAGNÓSTICO DE SUPUESTOS OLS")
print("=" * 60)
print(f"\nBreusch-Pagan (heterocedasticidad):")
print(f"  LM={bp_lm:.4f}  p={bp_p:.4f}")
if bp_p < 0.05:
    print("  ⚠️  HETEROCEDASTICIDAD DETECTADA (p < 0.05)")
    heterocedasticidad = True
else:
    print("  ✅ Homocedasticidad (p >= 0.05)")
    heterocedasticidad = False

print(f"\nTest de White (heterocedasticidad):")
print(f"  LM={w_lm:.4f}  p={w_p:.4f}")
print(f"  → {'⚠️  Heterocedasticidad' if w_p < 0.05 else '✅ Homocedasticidad'}")

print(f"\nDurbin-Watson (autocorrelación):")
print(f"  DW={dw:.4f}  → {'✅ Sin autocorrelación' if 1.5 < dw < 2.5 else '⚠️  Posible autocorrelación'}")

print(f"\nShapiro-Wilk (normalidad residuos):")
print(f"  W={sw_stat:.4f}  p={sw_p:.4f}  → {'✅ Normal' if sw_p > 0.05 else '⚠️  No normal'}")
print("=" * 60)

In [ ]:
# Gráficas de diagnóstico
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Residuos vs Ajustados
axes[0,0].scatter(y_pred, residuos, alpha=0.35, s=15, color='steelblue')
axes[0,0].axhline(0, color='red', linestyle='--')
axes[0,0].set_xlabel('Valores ajustados')
axes[0,0].set_ylabel('Residuos')
axes[0,0].set_title('Residuos vs Ajustados')

# 2. Q-Q Plot
sm.qqplot(residuos, line='s', ax=axes[0,1], alpha=0.5)
axes[0,1].set_title('Q-Q Plot (Normalidad de Residuos)')

# 3. Histograma de residuos
axes[1,0].hist(residuos, bins=30, color='steelblue', alpha=0.75, edgecolor='white', density=True)
xmin, xmax = axes[1,0].get_xlim()
x_norm = np.linspace(xmin, xmax, 100)
axes[1,0].plot(x_norm, stats.norm.pdf(x_norm, residuos.mean(), residuos.std()),
               'r-', linewidth=2, label='Normal teórica')
axes[1,0].set_title('Distribución de Residuos')
axes[1,0].legend()

# 4. Scale-Location (detecta heterocedasticidad visualmente)
sqrt_abs_res = np.sqrt(np.abs(residuos))
axes[1,1].scatter(y_pred, sqrt_abs_res, alpha=0.35, s=15, color='steelblue')
axes[1,1].set_xlabel('Valores ajustados')
axes[1,1].set_ylabel('√|Residuos|')
axes[1,1].set_title('Scale-Location (Homocedasticidad)')

plt.suptitle('Diagnósticos OLS Estándar', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{GRAFICAS}/ols_diagnosticos.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\n{'⚠️  Se aplicará corrección de heterocedasticidad (HC3 + WLS)' if heterocedasticidad else '✅ No se requiere corrección'}")

## 4. OLS con errores robustos HC3 (corrección de White)

Cuando hay heterocedasticidad, los errores estándar del OLS clásico son incorrectos.
HC3 (Heteroskedasticity-Consistent Covariance) corrige los errores estándar sin cambiar
los coeficientes, haciendo los tests t y F válidos.

In [ ]:
modelo_hc3 = sm.OLS(y, X_const).fit(cov_type='HC3')
print("=" * 60)
print("  OLS CON ERRORES ROBUSTOS HC3")
print("=" * 60)
print(modelo_hc3.summary())

# Comparar errores estándar OLS vs HC3
comp = pd.DataFrame({
    'Variable':    X_const.columns,
    'Coef':        modelo_ols.params.round(6),
    'SE_OLS':      modelo_ols.bse.round(6),
    'SE_HC3':      modelo_hc3.bse.round(6),
    'p_OLS':       modelo_ols.pvalues.round(4),
    'p_HC3':       modelo_hc3.pvalues.round(4),
})
print("\n=== Comparación de Errores Estándar OLS vs HC3 ===")
print(comp.to_string(index=False))

## 5. WLS — Mínimos Cuadrados Ponderados

WLS pondera cada observación inversamente proporcional a la varianza estimada
de sus residuos, corrigiendo la heterocedasticidad directamente en la estimación.

In [ ]:
# Estimar pesos: 1 / varianza estimada de los residuos
# Usamos |residuos OLS| como proxy de la desviación estándar local
abs_res = np.abs(modelo_ols.resid)

# Ajustar un modelo auxiliar para estimar la varianza
modelo_aux = sm.OLS(np.log(abs_res + 1e-8), X_const).fit()
sigma_hat  = np.exp(modelo_aux.fittedvalues)
pesos      = 1.0 / (sigma_hat ** 2)
pesos      = pesos / pesos.mean()  # normalizar

modelo_wls = sm.WLS(y, X_const, weights=pesos).fit()
print("=" * 60)
print("  WLS — MÍNIMOS CUADRADOS PONDERADOS")
print("=" * 60)
print(modelo_wls.summary())

## 6. Evaluación con 3 splits: 80/20 · 70/30 · 60/40

In [ ]:
def evaluar_modelo(nombre, y_true, y_pred, split):
    return {
        "Modelo": nombre,
        "Split":  split,
        "R²":     round(r2_score(y_true, y_pred), 4),
        "RMSE":   round(np.sqrt(mean_squared_error(y_true, y_pred)), 4),
        "MAE":    round(mean_absolute_error(y_true, y_pred), 4),
    }

resultados = []
splits = [("80/20", 0.20), ("70/30", 0.30), ("60/40", 0.40)]

for nombre_split, ts in splits:
    idx_train, idx_test = train_test_split(
        np.arange(len(X_raw)), test_size=ts, random_state=42)

    X_tr = X_raw.iloc[idx_train]
    X_te = X_raw.iloc[idx_test]
    y_tr = y.iloc[idx_train]
    y_te = y.iloc[idx_test]

    X_tr_c = sm.add_constant(X_tr)
    X_te_c = sm.add_constant(X_te)

    # ── OLS estándar ──────────────────────────────────────────────────────────
    m_ols = sm.OLS(y_tr, X_tr_c).fit()
    resultados.append(evaluar_modelo("OLS", y_te, m_ols.predict(X_te_c), nombre_split))

    # ── OLS HC3 ───────────────────────────────────────────────────────────────
    m_hc3 = sm.OLS(y_tr, X_tr_c).fit(cov_type='HC3')
    resultados.append(evaluar_modelo("OLS-HC3", y_te, m_hc3.predict(X_te_c), nombre_split))

    # ── WLS ───────────────────────────────────────────────────────────────────
    abs_r_tr = np.abs(m_ols.resid)
    m_aux    = sm.OLS(np.log(abs_r_tr + 1e-8), X_tr_c).fit()
    sigma_tr = np.exp(m_aux.predict(X_tr_c))
    w_tr     = 1.0 / (sigma_tr ** 2)
    w_tr     = w_tr / w_tr.mean()

    sigma_te = np.exp(m_aux.predict(X_te_c))
    w_te     = 1.0 / (sigma_te ** 2)

    m_wls = sm.WLS(y_tr, X_tr_c, weights=w_tr).fit()
    resultados.append(evaluar_modelo("WLS", y_te, m_wls.predict(X_te_c), nombre_split))

    print(f"\n=== Split {nombre_split} ===")
    print(f"  Train: {len(X_tr)} | Test: {len(X_te)}")
    for r in resultados[-3:]:
        print(f"  {r['Modelo']:8s} → R²={r['R²']:.4f}  RMSE={r['RMSE']:.4f}  MAE={r['MAE']:.4f}")

## 7. Gráficas comparativas por split

In [ ]:
df_res = pd.DataFrame(resultados)
colores = {"OLS": "#2196F3", "OLS-HC3": "#FF9800", "WLS": "#4CAF50"}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, met in zip(axes, ["R²", "RMSE", "MAE"]):
    for modelo, color in colores.items():
        sub = df_res[df_res["Modelo"] == modelo]
        ax.plot(sub["Split"], sub[met], marker='o', label=modelo,
                color=color, linewidth=2)
    ax.set_title(met, fontweight='bold')
    ax.set_xlabel("Split")
    ax.legend()

plt.suptitle("Comparativa OLS / OLS-HC3 / WLS por Split",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{GRAFICAS}/ols_comparativa_splits.png", dpi=150, bbox_inches='tight')
plt.show()

## 8. Diagnóstico post-corrección (WLS)

In [ ]:
res_wls = modelo_wls.resid

bp_lm2, bp_p2, _, _ = het_breuschpagan(res_wls, X_const)
dw2 = durbin_watson(res_wls)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(modelo_wls.fittedvalues, res_wls, alpha=0.35, s=15, color='#4CAF50')
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Valores ajustados WLS')
axes[0].set_ylabel('Residuos WLS')
axes[0].set_title('Residuos vs Ajustados — WLS')

sqrt_abs_wls = np.sqrt(np.abs(res_wls))
axes[1].scatter(modelo_wls.fittedvalues, sqrt_abs_wls, alpha=0.35, s=15, color='#4CAF50')
axes[1].set_xlabel('Valores ajustados WLS')
axes[1].set_ylabel('√|Residuos WLS|')
axes[1].set_title('Scale-Location — WLS')

plt.suptitle('Diagnóstico Post-Corrección (WLS)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{GRAFICAS}/wls_diagnosticos.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nBreusch-Pagan post-WLS: LM={bp_lm2:.4f}  p={bp_p2:.4f}")
print(f"  → {'✅ Heterocedasticidad neutralizada' if bp_p2 >= 0.05 else '⚠️  Heterocedasticidad residual'}")
print(f"Durbin-Watson WLS: {dw2:.4f}")

## 9. SUMMARY FINAL

In [ ]:
print("\n" + "=" * 70)
print("  SUMMARY FINAL — OLS / OLS-HC3 / WLS")
print("=" * 70)
print(df_res.to_string(index=False))
print("=" * 70)

for modelo in ["OLS", "OLS-HC3", "WLS"]:
    sub = df_res[df_res["Modelo"] == modelo]
    mejor = sub.loc[sub["R²"].idxmax()]
    print(f"\n{modelo}:")
    print(f"  Mejor R²   = {mejor['R²']:.4f} (Split {mejor['Split']})")
    print(f"  Menor RMSE = {sub['RMSE'].min():.4f} (Split {sub.loc[sub['RMSE'].idxmin(), 'Split']})")
    print(f"  Menor MAE  = {sub['MAE'].min():.4f} (Split {sub.loc[sub['MAE'].idxmin(), 'Split']})")

print("\n" + "=" * 70)
print("  DIAGNÓSTICO FINAL")
print("=" * 70)
print(f"  Breusch-Pagan OLS  : p={bp_p:.4f}  → {'⚠️  Heterocedasticidad' if bp_p < 0.05 else '✅ Homocedasticidad'}")
print(f"  Breusch-Pagan WLS  : p={bp_p2:.4f}  → {'✅ Neutralizada' if bp_p2 >= 0.05 else '⚠️  Residual'}")
print(f"  Test de White      : p={w_p:.4f}  → {'⚠️  Heterocedasticidad' if w_p < 0.05 else '✅ Homocedasticidad'}")
print(f"  Durbin-Watson OLS  : {dw:.4f}  → {'✅ Sin autocorrelación' if 1.5 < dw < 2.5 else '⚠️  Posible autocorrelación'}")
print(f"  Durbin-Watson WLS  : {dw2:.4f}  → {'✅ Sin autocorrelación' if 1.5 < dw2 < 2.5 else '⚠️  Posible autocorrelación'}")
print("=" * 70)

df_res